# Notebook 04 - Generation de Recettes

**MixCraft** - EFREI M1 DE&IA - Adam Beloucif & Emilien Morice - Janvier 2026

## Objectifs

1. Fine-tuner GPT-2 sur le corpus de recettes cocktails
2. Generer des recettes conditionnelles (par ingredients, par categorie)
3. Evaluer la qualite des generations via BLEU-4 et ROUGE-L
4. Comparer : generation templates vs GPT-2 fine-tune

In [ ]:
import sys; sys.path.insert(0, '..')
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

EFREI_NAVY = '#163767'
EFREI_PINK = '#FF43B8'

df = pd.read_csv('../data/processed/corpus_cocktails.csv') if Path('../data/processed/corpus_cocktails.csv').exists() else None
if df is None:
    from src.data_loader import load_all_datasets
    df = load_all_datasets()
print(f'Corpus : {len(df)} recettes')

## 1. Preparation du corpus de fine-tuning

In [ ]:
def format_for_training(row: pd.Series) -> str:
    """Formate une recette pour le fine-tuning GPT-2."""
    return (
        f"Recipe: {row.get('name', 'Unknown Cocktail')}\n"
        f"Category: {row.get('category', '')}\n"
        f"Ingredients: {row.get('ingredients', '')}\n"
        f"Instructions: {row.get('instructions', '')}\n"
        "<|endoftext|>"
    )

training_texts = df.apply(format_for_training, axis=1).tolist()

print(f'Textes d\'entrainement : {len(training_texts)}')
print(f'Exemple (tronque) :\n{training_texts[0][:400]}')

In [ ]:
# Statistiques sur les sequences d'entrainement
lengths = [len(t.split()) for t in training_texts]
print(f'Longueur moyenne  : {np.mean(lengths):.0f} mots')
print(f'Longueur mediane  : {np.median(lengths):.0f} mots')
print(f'Longueur max      : {max(lengths)} mots')
print(f'Longueur min      : {min(lengths)} mots')

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(lengths, bins=30, color=EFREI_NAVY, edgecolor='white', alpha=0.9)
ax.axvline(np.mean(lengths), color=EFREI_PINK, linestyle='--', linewidth=2,
           label=f'Moyenne : {np.mean(lengths):.0f} mots')
ax.set_title('Distribution des longueurs de sequences (fine-tuning GPT-2)', fontweight='bold', color=EFREI_NAVY)
ax.set_xlabel('Nombre de mots')
ax.set_ylabel('Frequence')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../assets/fig_seq_lengths.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Fine-tuning GPT-2 (optionnel - necessite GPU ou ~30 min CPU)

In [ ]:
from src.generator import CocktailGenerator

generator = CocktailGenerator()

# Le fine-tuning est commente pour les runs sans GPU
# Decommenter sur Colab ou machine avec GPU :

# train_metrics = generator.train(
#     texts=training_texts,
#     epochs=3,
#     batch_size=4,     # reduire si OOM
#     lr=5e-5,
#     save=True,
# )
# print('Fine-tuning termine :', train_metrics)

print('Note : fine-tuning desactive. Utilisation de GPT-2 de base ou mode template.')
print('Pour lancer : decommenter la cellule train sur une machine avec >=8GB RAM.')

## 3. Generation de recettes

In [ ]:
# Generation par template (toujours disponible)
prompts = [
    ['vodka', 'lime juice', 'ginger beer', 'mint'],
    ['rum', 'coconut cream', 'pineapple juice'],
    ['gin', 'tonic', 'cucumber', 'elderflower'],
    ['tequila', 'lime', 'salt', 'orange liqueur'],
]

print('=== Exemples de generation par template ===\n')
for ingredients in prompts:
    print(f'Ingredients : {ingredients}')
    try:
        recipes = generator.generate(ingredients, max_new_tokens=150, temperature=0.8)
        print('Recette generee :')
        print(recipes[0][:300])
    except Exception as e:
        # Fallback si modele pas disponible
        print(f'  [Mode template - {e}]')
        ings_str = ', '.join(ingredients)
        print(f"  Recette avec {ings_str} : Combiner dans un shaker avec glace.")
        print(f"  Agiter 15s, filtrer dans un verre adapte.")
    print('-' * 50)

## 4. Evaluation BLEU et ROUGE

In [ ]:
from src.generator import evaluate_generation

# Jeu de test : comparer generation avec references connues
test_pairs = [
    {
        'ingredients': ['vodka', 'lime', 'ginger beer'],
        'reference': 'Moscow Mule: Mix vodka and lime juice. Top with ginger beer. Garnish with mint and lime.'
    },
    {
        'ingredients': ['rum', 'mint', 'lime', 'sugar', 'soda'],
        'reference': 'Mojito: Muddle mint and lime with sugar. Add rum and ice. Top with soda water.'
    },
    {
        'ingredients': ['gin', 'lemon', 'sugar syrup'],
        'reference': 'Gin Sour: Shake gin, lemon juice and sugar syrup with ice. Strain into glass.'
    },
]

bleu_scores = []
rouge_scores = []

for pair in test_pairs:
    ings = pair['ingredients']
    ref = pair['reference']
    
    # Generation
    try:
        gen = generator.generate(ings, max_new_tokens=100)[0]
    except:
        gen = f"Mix {', '.join(ings)}. Shake with ice. Strain and serve."
    
    metrics = evaluate_generation(gen, ref)
    bleu_scores.append(metrics.get('bleu4', 0.0))
    rouge_scores.append(metrics.get('rouge_l', 0.0))
    print(f'Ingredients : {ings}')
    print(f'  BLEU-4   : {metrics.get("bleu4", 0):.4f}')
    print(f'  ROUGE-L  : {metrics.get("rouge_l", 0):.4f}')
    print()

print(f'Moyenne BLEU-4  : {np.mean(bleu_scores):.4f}')
print(f'Moyenne ROUGE-L : {np.mean(rouge_scores):.4f}')

In [ ]:
# Graphique des scores
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

recipe_names = [', '.join(p['ingredients'][:2]) for p in test_pairs]

axes[0].bar(range(len(bleu_scores)), bleu_scores, color=EFREI_NAVY, edgecolor='white')
axes[0].set_title('Scores BLEU-4 par recette', fontweight='bold', color=EFREI_NAVY)
axes[0].set_xticks(range(len(bleu_scores)))
axes[0].set_xticklabels(recipe_names, rotation=15, ha='right', fontsize=9)
axes[0].set_ylabel('BLEU-4')
axes[0].axhline(np.mean(bleu_scores), color=EFREI_PINK, linestyle='--',
                label=f'Moyenne : {np.mean(bleu_scores):.3f}')
axes[0].legend()

axes[1].bar(range(len(rouge_scores)), rouge_scores, color=EFREI_BLUE, edgecolor='white')
axes[1].set_title('Scores ROUGE-L par recette', fontweight='bold', color=EFREI_NAVY)
axes[1].set_xticks(range(len(rouge_scores)))
axes[1].set_xticklabels(recipe_names, rotation=15, ha='right', fontsize=9)
axes[1].set_ylabel('ROUGE-L')
axes[1].axhline(np.mean(rouge_scores), color=EFREI_PINK, linestyle='--',
                label=f'Moyenne : {np.mean(rouge_scores):.3f}')
axes[1].legend()

for ax in axes:
    ax.grid(True, axis='y', alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('../assets/fig_generation_metrics.png', dpi=150, bbox_inches='tight')
plt.show()